# Время поездки домой в Лондоне

Предсказываем время поездки по одному синтетическому CSV. `...` обозначает пропуск в коде.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### 0. Подготовка данных

Выполните ячейку, чтобы создать CSV.

In [ ]:
rng = np.random.default_rng(42)
data_dir = Path("seminar_data")
data_dir.mkdir(exist_ok=True)

n = 360
home_zone = rng.integers(1, 7, size=n)
distance_m = np.rint(
    np.clip(1800 + 2800 * (home_zone - 1) + rng.normal(0, 1500, n), 700, 26000)
).astype(int)
walk_pct = rng.integers(4, 30, size=n)
walk_m_truth = distance_m * walk_pct / 100
transfers = rng.integers(0, 4, n)
rush_hour = rng.binomial(1, 0.45, n)
transport = rng.choice(["метро", "автобус", "поезд"], n, p=[0.46, 0.34, 0.20])

# Условные зависимости для упражнения, время не взято из расписания.
transport_minutes = pd.Series(transport).map({"метро": 0, "автобус": 10, "поезд": -5})
travel_time = np.maximum(
    5, 7 + 1.8 * distance_m / 1000 + 1.7 * (home_zone - 1)
    + 0.008 * walk_m_truth + 4.5 * transfers + 5 * rush_hour
    + transport_minutes + rng.normal(0, 3, n)
).round(1).to_numpy()
travel_time[[7, 61, 197, 310]] = [-12, 480, 720, 365]
travel_time[143] = 112

use_km = rng.random(n) < 0.5
distance_value = np.where(use_km, distance_m / 1000, distance_m)
distance_unit = pd.Series(np.where(use_km, "км", "м"), dtype="string")
missing_unit = rng.choice(n, size=70, replace=False)
distance_unit.iloc[missing_unit] = pd.NA

pd.DataFrame({
    "distance_value": distance_value,
    "distance_unit": distance_unit,
    "walk_pct": walk_pct,
    "home_zone": home_zone,
    "transport": transport,
    "transfers": transfers,
    "rush_hour": rush_hour,
    "travel_time_min": travel_time,
}).to_csv(data_dir / "commute.csv", index=False)

## 1. Знакомимся с выгрузкой

Одна строка — поездка из центра Лондона. Таргет — `travel_time_min`, время в минутах. `home_zone` — зона дома, `transport` — основной вид транспорта, `transfers` — число пересадок, `rush_hour` — час пик (0 или 1).

In [ ]:
df = pd.read_csv(data_dir / "commute.csv")
display(df.head(8))

In [ ]:
display(df["travel_time_min"].describe().round(1))

## 2. Расстояние и пешая часть

`distance_value` — длина маршрута, `distance_unit` — м или км. Если единица пропущена, значения меньше 30 считаем километрами, остальные — метрами.

`walk_pct` — процент маршрута пешком. Добавьте `distance_m` и `walk_m` в метрах.

In [ ]:
in_km = ...
df["distance_m"] = ...
df["walk_m"] = ...

In [ ]:
display(df.loc[df["distance_unit"].isna(),
               ["distance_value", "distance_unit", "distance_m", "walk_pct", "walk_m"]].head(8))

## 3. Проверяем выбросы

Отмечаем подозрительные поездки правилом 1,5 IQR.

In [ ]:
duration = df["travel_time_min"]
q1, q3 = ...
iqr = ...
lower, upper = ...
print(f"Границы IQR: от {lower:.1f} до {upper:.1f} мин")

In [ ]:
suspect = (duration < lower) | (duration > upper)
display(df.loc[suspect].sort_values("travel_time_min"))

Для модели берём поездки длительностью больше нуля и до 180 минут включительно.

In [ ]:
invalid = ...
df_valid = ...
print("Удалили строк:", invalid.sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)
axes[0].scatter(df["distance_m"] / 1000, duration, s=13, alpha=0.5)
axes[0].scatter(
    df.loc[invalid, "distance_m"] / 1000, duration[invalid],
    s=34, color="#cc4928", label="Удаляем по правилу задачи",
)
axes[0].legend()
axes[0].set_title("До удаления")
axes[1].scatter(df_valid["distance_m"] / 1000, df_valid["travel_time_min"], s=13, alpha=0.5)
axes[1].set_title("После удаления")
for ax in axes:
    ax.set(xlabel="Длина маршрута, км", ylabel="Время в пути, мин")
fig.tight_layout()
plt.show()

## 4. Признаки

Зона остаётся числом, транспорт кодируем как категорию.

In [ ]:
numeric_features = ["distance_m", "walk_m", "home_zone", "transfers", "rush_hour"]
features = numeric_features + ["transport"]
target_column = "travel_time_min"

display(df_valid.groupby("home_zone")["distance_m"].agg(["count", "median"]).round(0))
display(df_valid["transport"].value_counts())

## 5. Подготовка признаков и обучение

Разбиение 80/20, `random_state=42`.

In [ ]:
X = df_valid[features]
y = df_valid[target_column]
X_train, X_test, y_train, y_test = ...
print("Обучение:", len(X_train), "| Тест:", len(X_test))

### Масштабируем числовые признаки

In [ ]:
scaler = StandardScaler()
X_train_scaled = ...
X_test_scaled = ...

In [ ]:
display(pd.DataFrame({
    "До": X_train[numeric_features].std(ddof=0),
    "После": X_train_scaled.std(axis=0),
}).round(2))

### Кодируем вид транспорта

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore")
X_train_encoded = ...
X_test_encoded = ...

In [ ]:
display(pd.DataFrame(
    X_train_encoded[:5].astype(int),
    columns=encoder.get_feature_names_out(),
    index=X_train.index[:5],
))

### Собираем признаки и обучаем линейную регрессию

In [ ]:
X_train_ready = ...
X_test_ready = ...

model = LinearRegression()
# ... обучение модели

## 6. MAE, MSE и MAPE

MAPE возвращаем в процентах.

In [ ]:
y_true = y_test.to_numpy()
y_pred = model.predict(X_test_ready)
display(pd.DataFrame({"Факт, мин": y_true, "Прогноз, мин": y_pred}, index=X_test.index).head().round(1))

In [ ]:
def mae(y, pred):
    return ...


def mse(y, pred):
    return ...


def mape(y, pred):
    return ...

In [ ]:
def evaluate(actual, prediction):
    return {
        "MAE, мин": mae(actual, prediction),
        "MSE, мин²": mse(actual, prediction),
        "MAPE, %": mape(actual, prediction),
    }


baseline = np.full(len(y_true), y_train.median())
display(pd.DataFrame({
    "Медиана train": evaluate(y_true, baseline),
    "Линейная регрессия": evaluate(y_true, y_pred),
}).T.round(2))

## 7. Влияние выбросов

Сравниваем обучение с ошибочными строками и без них на одном тесте.

Допишите функцию повторного обучения и предсказания.

In [ ]:
def fit_and_predict(X_train, y_train, X_eval):
    return ...

In [ ]:
bad_train = df.loc[invalid, features + [target_column]]
X_noisy = pd.concat([X_train, bad_train[features]])
y_noisy = pd.concat([y_train, bad_train[target_column]])
assert X_noisy.index.intersection(X_test.index).empty

y_pred_noisy = fit_and_predict(X_noisy, y_noisy, X_test)

In [ ]:
display(pd.DataFrame({
    "С ошибочными строками": evaluate(y_true, y_pred_noisy),
    "После проверки": evaluate(y_true, y_pred),
}).T.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.scatter(y_true, y_pred_noisy, s=21, alpha=0.65, label="С ошибочными строками")
ax.scatter(y_true, y_pred, s=21, alpha=0.65, label="После проверки")
bounds = [y_true.min(), y_true.max()]
ax.plot(bounds, bounds, "--", color="black", linewidth=1, label="Точный прогноз")
ax.set(xlabel="Фактическое время, мин", ylabel="Прогноз, мин")
ax.legend()
fig.tight_layout()
plt.show()

## Если осталось время

### MAE и MSE: две прямые на одних и тех же данных

In [ ]:
X_distance = X_noisy[["distance_m"]].to_numpy() / 1000

model_mse = LinearRegression().fit(X_distance, y_noisy)
model_mae = SGDRegressor(
    loss="epsilon_insensitive", epsilon=0, penalty=None,
    learning_rate="constant", eta0=0.001,
    max_iter=3000, tol=None, average=True, random_state=42,
).fit(X_distance, y_noisy)

In [ ]:
distance_line = np.linspace(X_distance.min(), X_distance.max(), 100).reshape(-1, 1)
bad_mask = X_noisy.index.isin(bad_train.index)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for ax in axes:
    ax.scatter(X_distance[~bad_mask, 0], y_noisy[~bad_mask], s=14, alpha=0.35,
               color="gray", label="Маршруты")
    ax.scatter(X_distance[bad_mask, 0], y_noisy[bad_mask], s=35,
               color="#992846", label="Ошибочные записи")
    ax.plot(distance_line[:, 0], model_mse.predict(distance_line), color="#bd4919",
            linewidth=2, label="MSE")
    ax.plot(distance_line[:, 0], model_mae.predict(distance_line), color="#146579",
            linestyle="--", linewidth=2, label="MAE")
    ax.set(xlabel="Длина маршрута, км", ylabel="Время в пути, мин")

axes[0].set_title("Все обучающие точки")
axes[0].legend()
axes[1].set_title("Те же прямые крупным планом")
axes[1].set_ylim(0, y_train.max() + 5)
fig.tight_layout()
plt.show()

### Коэффициенты основной модели

In [ ]:
weights = pd.DataFrame({
    "признак": numeric_features,
    "шаг": ["1 км", "100 м пешком", "1 зона", "1 пересадка", "переход к часу пик"],
    "изменение прогноза, мин": model.coef_[:len(numeric_features)] / scaler.scale_
    * np.array([1000, 100, 1, 1, 1]),
})
display(weights.round(2))

### Кросс-валидация

In [ ]:
cv_scores = []
folds = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_ids, validation_ids) in enumerate(folds.split(X_train), start=1):
    validation_pred = fit_and_predict(
        X_train.iloc[train_ids], y_train.iloc[train_ids], X_train.iloc[validation_ids]
    )
    cv_scores.append({
        "Фолд": fold,
        **evaluate(y_train.iloc[validation_ids].to_numpy(), validation_pred),
    })

fold_results = pd.DataFrame(cv_scores).set_index("Фолд")
display(fold_results.round(2))
display(fold_results.mean().rename("Среднее по фолдам").round(2))